In [1]:
# Parameters
DB_PATH                   = "../../../DB/oedb_baseline_v3.db"
BENCHMARK_PATH            = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET           = "merged_answers"
MATCHED_PARTICIPANTS_CSV  = "matched_participants.csv"
MATCHED_QUESTIONS_CSV     = "matched_questions.csv"
MATCHED_CSV_PATH          = "matched_answers.csv"
NOTEGROUP_ID_MIN          = 1
NOTEGROUP_ID_MAX          = 23
MATCH_THRESHOLD           = 85   # score scale is 0-100

WEIGHT_CONTENT     = 0.8
WEIGHT_PARTICIPANT = 0.1
WEIGHT_QUESTION     = 0.1

In [2]:
import sqlite3
import re
import pandas as pd
from rapidfuzz import fuzz

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT answerID, notegroupID, questionID, participantID,
                  answer_content_oriLAN, answer_content_EN
           FROM answers
           WHERE notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    df["answerID"] = df["answerID"].astype(int)
    return df.set_index("answerID")

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df["answerID"]    = df["answerID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("answerID")

def load_id_map(csv_path, etl_col, bm_col):
    """Load a matched-pairs CSV and return etl->bm and bm->etl id dicts."""
    df = pd.read_csv(csv_path)
    df[etl_col] = df[etl_col].astype(int)
    df[bm_col]  = df[bm_col].astype(int)
    etl_to_bm = dict(zip(df[etl_col], df[bm_col]))
    bm_to_etl = dict(zip(df[bm_col], df[etl_col]))
    return etl_to_bm, bm_to_etl

etl = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm  = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

participant_etl_to_bm, participant_bm_to_etl = load_id_map(
    MATCHED_PARTICIPANTS_CSV, "etl_participantID", "bm_participantID"
)
question_etl_to_bm, question_bm_to_etl = load_id_map(
    MATCHED_QUESTIONS_CSV, "etl_questionID", "bm_questionID"
)

print("ETL records:      ", len(etl))
print("Benchmark records:", len(bm))
print("Matched participants:", len(participant_etl_to_bm))
print("Matched questions:   ", len(question_etl_to_bm))

ETL records:       1222
Benchmark records: 1221
Matched participants: 93
Matched questions:    418


In [3]:
def normalise_str(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    s = str(val).strip().lower()
    s = re.sub(r'\s*\n\s*', '\n', s)
    return s

def normalise_id(val):
    """Normalise a participantID/questionID value to int, or None."""
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    try:
        return int(float(str(val).strip()))
    except (ValueError, TypeError):
        return None

def participant_match(etl_row, bm_row):
    """
    Return 1 if ETL participantID, mapped to BM space, equals BM row's participantID,
    or if both sides are null. Else 0.
    """
    etl_pid = normalise_id(etl_row.get("participantID"))
    bm_pid  = normalise_id(bm_row.get("participantID"))
    if etl_pid is None and bm_pid is None:
        return 1
    mapped_pid = participant_etl_to_bm.get(etl_pid) if etl_pid is not None else None
    return 1 if mapped_pid is not None and mapped_pid == bm_pid else 0

def question_match(etl_row, bm_row):
    """
    Return 1 if ETL questionID, mapped to BM space, equals BM row's questionID,
    or if both sides are null. Else 0.
    """
    etl_qid = normalise_id(etl_row.get("questionID"))
    bm_qid  = normalise_id(bm_row.get("questionID"))
    if etl_qid is None and bm_qid is None:
        return 1
    mapped_qid = question_etl_to_bm.get(etl_qid) if etl_qid is not None else None
    return 1 if mapped_qid is not None and mapped_qid == bm_qid else 0

def safe_ratio(a, b):
    if a is None or b is None:
        return None
    return fuzz.ratio(a, b)

def content_score(etl_row, bm_row):
    """
    Compare answer_content_oriLAN / answer_content_EN, allowing for the
    possibility that the two fields are cross-matched (swapped).
    Returns the highest single-field similarity score across all four
    possible pairings (0-100). Null pairs are excluded.
    """
    etl_ori = normalise_str(etl_row.get("answer_content_oriLAN"))
    etl_en  = normalise_str(etl_row.get("answer_content_EN"))
    bm_ori  = normalise_str(bm_row.get("answer_content_oriLAN"))
    bm_en   = normalise_str(bm_row.get("answer_content_EN"))

    scores = [
        safe_ratio(etl_ori, bm_ori),
        safe_ratio(etl_en,  bm_en),
        safe_ratio(etl_ori, bm_en),
        safe_ratio(etl_en,  bm_ori),
    ]
    scores = [s for s in scores if s is not None]

    return max(scores) if scores else 0.0

def answer_pair_score(etl_row, bm_row):
    p_score = participant_match(etl_row, bm_row) * 100
    q_score = question_match(etl_row, bm_row) * 100
    c_score = content_score(etl_row, bm_row)
    return WEIGHT_PARTICIPANT * p_score + WEIGHT_QUESTION * q_score + WEIGHT_CONTENT * c_score

def is_substring_match(etl_row, bm_row, threshold):
    """
    Secondary match condition: BM answer_content_oriLAN closely aligns with
    a portion of the ETL text (handles merged/interrupted answers), AND
    supporting fields (participant_match, question_match) both equal 1.
    """
    e = normalise_str(etl_row.get("answer_content_oriLAN"))
    b = normalise_str(bm_row.get("answer_content_oriLAN"))
    if e is None or b is None:
        return False

    # partial_ratio finds the best-aligned substring match, tolerant of
    # interruptions like inserted speaker tags
    if fuzz.partial_ratio(e, b) < threshold:
        return False

    p_match = participant_match(etl_row, bm_row)
    q_match = question_match(etl_row, bm_row)

    return p_match == 1 and q_match == 1

In [4]:
def map_answers(etl_df, bm_df, threshold):
    """
    Match answers within each notegroupID by weighted similarity:
    80% content similarity (direct or cross oriLAN/EN alignment),
    10% participant identity match, 10% question identity match (via prior mappings).
    Returns:
        matched  : list of (etl_idx, bm_idx, score)
        etl_only : list of etl_idx  -> FP rows
        bm_only  : list of bm_idx   -> FN rows
    """
    matched  = []
    etl_only = []
    bm_only  = []

    for ng_id in sorted(etl_df["notegroupID"].unique()):
        etl_ng = etl_df[etl_df["notegroupID"] == ng_id]
        bm_ng  = bm_df[bm_df["notegroupID"]  == ng_id]

        if bm_ng.empty:
            etl_only.extend(etl_ng.index.tolist())
            continue
        if etl_ng.empty:
            bm_only.extend(bm_ng.index.tolist())
            continue

        scores = {}
        for ei in etl_ng.index:
            for bi in bm_ng.index:
                primary_score = answer_pair_score(etl_ng.loc[ei], bm_ng.loc[bi])
                if primary_score < threshold and is_substring_match(etl_ng.loc[ei], bm_ng.loc[bi], 85):
                    primary_score = threshold
                scores[(ei, bi)] = primary_score

        used_etl = set()
        used_bm  = set()
        for (ei, bi), score in sorted(scores.items(), key=lambda x: -x[1]):
            if score < threshold:
                break
            if ei in used_etl or bi in used_bm:
                continue
            matched.append((ei, bi, round(score, 2)))
            used_etl.add(ei)
            used_bm.add(bi)

        etl_only.extend([i for i in etl_ng.index if i not in used_etl])
        bm_only.extend( [i for i in bm_ng.index  if i not in used_bm])

    return matched, etl_only, bm_only


matched, etl_only, bm_only = map_answers(etl, bm, MATCH_THRESHOLD)

print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

Matched pairs : 1194
ETL-only (FP) : 28
BM-only  (FN) : 27


In [6]:
print("=== ETL-only (FP) ===")
display(etl.loc[etl_only, ["notegroupID", "participantID", "questionID", "answer_content_oriLAN", "answer_content_EN"]])

print("\n=== BM-only (FN) ===")
bm_cols = [c for c in ["notegroupID", "participantID", "questionID", "answer_content_oriLAN", "answer_content_EN"] if c in bm.columns]
display(bm.loc[bm_only, bm_cols])

=== ETL-only (FP) ===


,notegroupID,participantID,questionID,answer_content_oriLAN,answer_content_EN
answerID,,,,,
432,6,36.0,135,"What is needed to improve this? (examples: motivation, r...",
590,9,45.0,194,-,
594,9,45.0,195,-,
598,9,45.0,196,,
606,9,45.0,201,-,
610,9,45.0,202,,
611,9,46.0,202,,
613,9,48.0,202,,
614,9,45.0,203,,



=== BM-only (FN) ===


,notegroupID,participantID,questionID,answer_content_oriLAN,answer_content_EN
answerID,,,,,
189,3,24,40,"they taught us, Dutch language. We took certificate. The...",NaN
1224,3,24,40,The most helpful is network. Gemeente.,NaN
1225,3,27,40,"Telegram can help, but I dont know much about them. But ...",NaN
1226,3,NaN,40,Awal ma tefta7 mawdou3 elshoghl mas2oul el COA be7ki ma ...,NaN
1227,3,26,48,"if you dont accept the others, it’ll be challenging beca...",NaN
1228,5,34,106,Racist people here.,NaN
1229,11,51,261,"Den Helder met afstand naar ons.""\n""Ik mis mijn huis eno...",NaN
1230,11,51,266,Ik wil een eigen woning – omdat dat een basisrecht is. M...,NaN
1231,11,52,266,"Ik wil me écht veilig voelen – niet alleen fysiek, maar ...",NaN


In [7]:
csv_rows = []
for ei, bi, score in matched:
    csv_rows.append({
        "notegroupID":   etl.loc[ei, "notegroupID"],
        "etl_answerID":  ei,
        "bm_answerID":   bi,
        "etl_oriLAN":    etl.loc[ei, "answer_content_oriLAN"],
        "bm_oriLAN":     bm.loc[bi, "answer_content_oriLAN"] if "answer_content_oriLAN" in bm.columns else None,
        "etl_EN":        etl.loc[ei, "answer_content_EN"],
        "bm_EN":         bm.loc[bi, "answer_content_EN"] if "answer_content_EN" in bm.columns else None,
        "match_score":   score,
    })

matched_csv = pd.DataFrame(csv_rows)
matched_csv.to_csv(MATCHED_CSV_PATH, index=False)
print(f"Saved {len(matched_csv)} matched pairs to {MATCHED_CSV_PATH}")
matched_csv.head(3)

Saved 1194 matched pairs to matched_answers.csv


,notegroupID,etl_answerID,bm_answerID,etl_oriLAN,bm_oriLAN,etl_EN,bm_EN,match_score
0,1,1,1,Not good in general. The school is slow and its educatio...,Not good in general. The school is slow and its educatio...,,NaN,100.0
1,1,2,2,It’s not as I hoped. I expected the language study perio...,It’s not as I hoped. I expected the language study perio...,,NaN,100.0
2,1,3,3,"For me, things are not going well. I don’t want to be ne...","For me, things are not going well. I don’t want to be ne...",,NaN,100.0
